# Data acquisition and preprocessing

Generated notebook for the GridMind IS2 project. Run from the project root or use the first cell to configure the path.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'config.py').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from importlib import import_module
import pandas as pd
source=import_module('01_data_acquisition.data_source')
prep=import_module('02_data_preprocessing.preprocess')

 Reproducible procedural data; explicitly synthetic, generated from schedules + noise.
raw=source.generate_building_data(days=60,seed=42,interval_minutes=60)
raw.head()

In [ ]:
print('Rows:',len(raw),'Synthetic source:',raw['is_synthetic'].all())
print(raw.isna().sum())
clean=prep.clean_and_engineer(raw)
train,validation,test=prep.chronological_split(clean)
print({k:len(v) for k,v in {'train':train,'validation':validation,'test':test}.items()})
assert train.timestamp.max()<validation.timestamp.min()<test.timestamp.min()

 Optional real-data alternative (requires internet). UCI supplies measured household load only.
# uci=source.load_uci_household(cache_dir=ROOT/'data', max_rows=300000)
# uci.head()

In [ ]:
import plotly.express as px
fig=px.line(clean.tail(24*7),x='timestamp',y=['load_kw','solar_kw'],title='One week: load and solar (synthetic covariates)')
fig.show()